<a href="https://colab.research.google.com/github/shqtbz143/BookAI/blob/main/T5_small_testipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets --quiet
!pip install accelerate -U --quiet


In [ ]:
#라이브러리 import
import json
import random
from datasets import Dataset, DatasetDict
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq

In [ ]:
# 데이터 로딩 -> 파일 업로드
from google.colab import files
uploaded = files.upload()


Saving 문단 - 질문 773.json to 문단 - 질문 773 (2).json


In [ ]:
with open('문단 - 질문 773.json', 'r', encoding='utf-8') as f:
    data = json.load(f)


In [ ]:
# 전처리: 질문 1개만 생성하도록 단순화 (question_1만 사용)
def preprocess(entry):
    input_text = f"generate question: {entry['paragraph'].strip()}"
    question_1 = entry.get("question_1", "").strip()
    return {"input": input_text, "output": question_1}

# question_1만 존재하는 항목 사용
processed_data = [preprocess(entry) for entry in data if entry.get("question_1")]


In [ ]:
#테스트로 데이터 2개 print
for i in range(2):
    print(f"Sample {i+1}")
    print(f"Input (Paragraph):\n{processed_data[i]['input']}\n")
    print(f"Output (Questions):\n{processed_data[i]['output']}\n")
    print("="*80)


Sample 1
Input (Paragraph):
generate question: 또한 여러 가지 주제뿐만 아니라 그 사이에서 갈등하고 성장하는 등장 인물들의 이야기, 다양한 책들의 인용문들을 보는 재미 또한 있어서 너무 자극적이지 않고 천천히, 조금씩 생각하며 읽어보기에 좋은 책이다. 그리 고, 소설 자체가 작가의 휴식처 혹은 아지트 같단 생각도 들었는데, 좋아 하는 것들과 바라는 것들로 꼼꼼히 채운 아지트를 자랑스럽게 소개하는 주인장처럼, 이 소설로 초대해서 작가가 사랑하는 것들과 그 따뜻한 휴식 처를 독자에게 소개하는 것 같은 느낌이 들었다. 이러한 이야기를 읽으면 서 나도 이처럼 내게 쉴 곳이 될 만한 휴식처를 언젠가 만들 수 있을런지 혹은 내 삶 속에서 찾을 수 있을런지에 대해 즐거운 상상을 해보는 좋은 계기가 되어준 책이었다.

Output (Questions):
당신만의 ‘작가의 아지트’ 같은 공간이나 활동이 있다면, 어디인가요? <sep> 좋아하는 것들로 가득 찬 삶을 상상할 때, 당신의 목록에는 무엇이 가장 먼저 올라올까요?

Sample 2
Input (Paragraph):
generate question: 하지만 이 또한 한쪽 면만 보고 판단할 수 없는 문제입니다. 사건을 보면 그 속의 사연을 알아야 하고 평면적인 정보만을 받아 비판할 것이 아니라, 입체적인 정보를 모두 받아들이고 이해해야 합니다.

Output (Questions):
한 사람이나 사건을 입체적으로 바라보는 시각을 기르기 위해 평소에 어떤 노력을 하고 계신가요? <sep> 누군가의 행동을 판단할 때 여러분은 어떤 정보나 관점을 가장 중요하게 여기시나요? 감정, 사실, 혹은 배경 중에서 말이에요.



In [ ]:
#90은 훈련, 10은 검증
# Dataset 생성
random.shuffle(processed_data)#섞기
train_size = int(len(processed_data) * 0.9)
train_data = processed_data[:train_size]
val_data = processed_data[train_size:]

#Hugging Face Dataset 객체 변
dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data)
})

In [ ]:
#모델, tokenizer 로
#Hugging Face 모델은 텍스트를 숫자(토큰)으로 바꿔야 함
#T5-small 버전
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

#tokenizer에 sep추가
tokenizer.add_tokens(["<sep>"])
model.resize_token_embeddings(len(tokenizer))



Embedding(32101, 512)

In [ ]:
#tokenization 함수 정의
#Hugging Face 모델은 텍스트를 숫자(토큰)로 바꿔야만 함
#tokenizer을 숫자 시퀀스로 변환하는 과정
MAX_INPUT = 512
MAX_OUTPUT = 128

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        max_length=MAX_INPUT,
        padding="max_length",
        truncation=True
    )
    labels = tokenizer(
        examples["output"],
        max_length=MAX_OUTPUT,
        padding="max_length",
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/695 [00:00<?, ? examples/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

In [ ]:
# 학습 설정
training_args = TrainingArguments(
   output_dir="./checkpoints/t5_paragraph_model",#drive 저장 x
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit = 2,
    load_best_model_at_end=True,
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.0,
    logging_dir="./logs",
    logging_steps=50,
    fp16=False,#True로하면 속도 down
    report_to="none"  # wandb 끄기
)
#런타임 계속 중지되면 save_strategy 단위를 steps으로 바꾸기


data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
#Trainer 정의 및 학습 시작
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


<ipython-input-77-f53833ac9584>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.244100,0.206651
2,0.215100,0.193313
3,0.202000,0.190597


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=522, training_loss=0.34612107459612734, metrics={'train_runtime': 5464.929, 'train_samples_per_second': 0.382, 'train_steps_per_second': 0.096, 'total_flos': 282187656069120.0, 'train_loss': 0.34612107459612734, 'epoch': 3.0})

In [ ]:
# 모델 훈련이 끝난 후
model.save_pretrained("/content/t5_paragraph2questions_model")
tokenizer.save_pretrained("/content/t5_paragraph2questions_model")


('/content/t5_paragraph2questions_model/tokenizer_config.json',
 '/content/t5_paragraph2questions_model/special_tokens_map.json',
 '/content/t5_paragraph2questions_model/spiece.model',
 '/content/t5_paragraph2questions_model/added_tokens.json')

In [ ]:
#평가 시작
!pip install evaluate bert_score --quiet
!pip install rouge_score --quiet


In [ ]:
from tqdm import tqdm
import evaluate

# 평가 지표 불러오기
meteor = evaluate.load("meteor")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

predictions = []
references = []

for example in tqdm(val_data):
    input_text = example["input"]  # 문단
    ref_text = example["output"]   # question_1 <sep> question_2

    # 모델 추론
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids
    output_ids = model.generate(input_ids, max_length=128)
    pred_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    predictions.append(pred_text)
    references.append([ref_text])  # ← 꼭 2차원 리스트로 (하나라도 리스트로 감싸야 함)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
100%|██████████| 78/78 [05:00<00:00,  3.85s/it]


In [ ]:
# METEOR
meteor_score = meteor.compute(predictions=predictions, references=references)
print(" METEOR:", round(meteor_score["meteor"], 4))

# ROUGE
rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references])
print("ROUGE-L:", round(rouge_score["rougeL"], 4))

# BERTScore
bert_score = bertscore.compute(predictions=predictions, references=[r[0] for r in references], lang="ko")
avg_bert_f1 = sum(bert_score["f1"]) / len(bert_score["f1"])
print(" BERTScore (F1 평균):", round(avg_bert_f1, 4))


 METEOR: 0.0283
ROUGE-L: 0.2051
 BERTScore (F1 평균): 0.1299


In [ ]:
print(tokenizer.decode(output_ids[0], skip_special_tokens=False))  # False로 설정해서 무슨 토큰 생성됐는지 보기


<pad> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>


In [ ]:
print(val_data[0]["input"])  # 문단
print(tokenizer(val_data[0]["input"]))  # 토큰화 결과


학교 도서관 한가운데에 정갈하게 비치된 올해의 책들 서가에서 나에게 유독 눈길이 가는 책이 있었다. 그 책의 제목은 바로 ‘밝은 밤’이었다. 원래 밤은 어둡고 깜깜한 법인데 왜 책 제목은 밝은 밤일까? 대체 무슨 의미가 있을까? 하는 궁금증을 안고 나는 이 책을 읽어보게 되었다.
{'input_ids': [3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 5, 3, 2, 3, 2, 3, 2, 3, 2, 458, 2, 3, 2, 22, 2, 5, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 58, 3, 2, 3, 2, 3, 2, 3, 2, 58, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
